## 1. Install dependencies

Run this once. If you're using the provided `requirements.txt`, this installs everything needed (including Jupyter kernel support).

In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
## 2. Imports

In [3]:
import os
import pickle
from pathlib import Path
from typing import List

import requests
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_groq import ChatGroq

load_dotenv()
print("✓ Imports OK")

C:\Users\damsara\AppData\Local\Temp\ipykernel_25280\1310979380.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✓ Imports OK


## 3. Environment setup

Create a `.env` file next to this notebook (if you don't have one yet) with:


In [5]:
for directory in ["./pdfs", "./vector_store"]:
    os.makedirs(directory, exist_ok=True)
    print(f"✓ Directory ready: {directory}")

missing = [k for k in ("GROQ_API_KEY", "JINA_API_KEY") if not os.getenv(k)]
if missing:
    print(f"⚠️  Missing environment variables: {', '.join(missing)}. Add them to your .env file before running the chatbot.")
else:
    print("✓ API keys found in environment")

✓ Directory ready: ./pdfs
✓ Directory ready: ./vector_store
✓ API keys found in environment


## 4. Configuration

(from the original `config.py`)

In [6]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
JINA_API_KEY = os.getenv("JINA_API_KEY")

PDF_FOLDER_PATH = os.getenv("PDF_FOLDER_PATH", "./pdfs")
VECTOR_DB_PATH = os.getenv("VECTOR_DB_PATH", "./vector_store")

JINA_MODEL = os.getenv("MODEL_NAME", "jina-embeddings-v3")
# NOTE: "mixtral-8x7b-32768" has been decommissioned on Groq.
# Default changed to a currently supported model; check
# https://console.groq.com/docs/models for the latest options.
GROQ_MODEL = os.getenv("LLM_MODEL", "llama-3.3-70b-versatile")

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
SEARCH_K = 5
TEMPERATURE = 0.7
MAX_TOKENS = 1024

TEXT_SPLITTER_CONFIG = {
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "separators": ["\n\n", "\n", " ", ""],
}

LLM_CONFIG = {
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS,
}

RETRIEVER_CONFIG = {
    "search_type": "similarity",
    "search_kwargs": {"k": SEARCH_K},
}

print(f"Embedding model: {JINA_MODEL}")
print(f"LLM model: {GROQ_MODEL}")

Embedding model: jina-embeddings-v3
LLM model: llama-3.3-70b-versatile


## 5. Jina Embeddings

(from the original `embeddings.py`)

In [7]:
class JinaEmbeddings(Embeddings):
    """Custom embeddings class for Jina Embeddings API"""

    def __init__(self):
        self.api_key = os.getenv("JINA_API_KEY")
        self.base_url = "https://api.jina.ai/v1"
        self.model = JINA_MODEL

        if not self.api_key:
            raise ValueError("JINA_API_KEY not found in environment variables")

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """Embed search docs"""
        return self._embed_batch(texts)

    def embed_query(self, text: str) -> List[float]:
        """Embed query text"""
        result = self._embed_batch([text])
        return result[0]

    def _embed_batch(self, texts: List[str]) -> List[List[float]]:
        """Embed a batch of texts using Jina API"""
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Accept-Encoding": "gzip",
        }

        data = {"model": self.model, "input": texts}

        try:
            response = requests.post(
                f"{self.base_url}/embeddings",
                headers=headers,
                json=data,
                timeout=30,
            )
            response.raise_for_status()
            result = response.json()
            embeddings = [item["embedding"] for item in result["data"]]
            return embeddings

        except requests.exceptions.RequestException as e:
            print(f"Error calling Jina API: {e}")
            raise

print("✓ JinaEmbeddings defined")

✓ JinaEmbeddings defined


## 6. PDF Loader

(from the original `pdf_loader.py`)

In [8]:
class PDFLoader:
    """Load and process PDF documents"""

    def __init__(self, pdf_folder_path: str = "./"):
        self.pdf_folder_path = pdf_folder_path
        self.text_splitter = RecursiveCharacterTextSplitter(**TEXT_SPLITTER_CONFIG)

    def load_pdfs(self) -> List[Document]:
        """Load all PDF files from the folder"""
        documents = []

        if not os.path.exists(self.pdf_folder_path):
            print(f"PDF folder not found at {self.pdf_folder_path}")
            return documents

        pdf_files = list(Path(self.pdf_folder_path).glob("*.pdf"))

        if not pdf_files:
            print(f"No PDF files found in {self.pdf_folder_path}")
            return documents

        print(f"Found {len(pdf_files)} PDF files. Loading...")

        for pdf_file in pdf_files:
            try:
                print(f"Loading {pdf_file.name}...")
                loader = PyPDFLoader(str(pdf_file))
                pages = loader.load()
                documents.extend(pages)
                print(f"✓ Loaded {len(pages)} pages from {pdf_file.name}")
            except Exception as e:
                print(f"✗ Error loading {pdf_file.name}: {e}")

        return documents

    def split_documents(self, documents: List[Document]) -> List[Document]:
        """Split documents into chunks"""
        if not documents:
            print("No documents to split")
            return []

        print(f"\nSplitting {len(documents)} documents into chunks...")
        split_docs = self.text_splitter.split_documents(documents)
        print(f"✓ Created {len(split_docs)} text chunks")

        return split_docs

    def process_pdfs(self) -> List[Document]:
        """Load and process all PDFs"""
        documents = self.load_pdfs()
        split_documents = self.split_documents(documents)
        return split_documents

print("✓ PDFLoader defined")

✓ PDFLoader defined


## 7. Vector Store

(from the original `vector_store.py`)

In [9]:
class VectorStore:
    """Manage vector store for RAG"""

    def __init__(self, vector_db_path: str = "./vector_store"):
        self.vector_db_path = vector_db_path
        self.embeddings = JinaEmbeddings()
        self.vector_store = None

    def create_vector_store(self, documents: List[Document]) -> FAISS:
        """Create vector store from documents"""
        if not documents:
            raise ValueError("No documents provided to create vector store")

        print("\nCreating vector store with Jina embeddings...")

        self.vector_store = FAISS.from_documents(documents, self.embeddings)

        print(f"✓ Vector store created with {len(documents)} documents")
        return self.vector_store

    def save_vector_store(self):
        """Save vector store to disk"""
        if self.vector_store is None:
            raise ValueError("No vector store to save")

        os.makedirs(self.vector_db_path, exist_ok=True)
        self.vector_store.save_local(self.vector_db_path)
        print(f"✓ Vector store saved to {self.vector_db_path}")

    def load_vector_store(self) -> FAISS:
        """Load vector store from disk"""
        if not os.path.exists(self.vector_db_path):
            raise ValueError(f"Vector store not found at {self.vector_db_path}")

        print(f"Loading vector store from {self.vector_db_path}...")
        # allow_dangerous_deserialization is required by recent FAISS/langchain
        # versions because loading a pickle index involves unpickling.
        # Only load vector stores you created yourself.
        self.vector_store = FAISS.load_local(
            self.vector_db_path,
            self.embeddings,
            allow_dangerous_deserialization=True,
        )
        print("✓ Vector store loaded")
        return self.vector_store

    def similarity_search(self, query: str, k: int = 5) -> List[Document]:
        """Search for similar documents"""
        if self.vector_store is None:
            raise ValueError("Vector store not loaded")

        results = self.vector_store.similarity_search(query, k=k)
        return results

print("✓ VectorStore defined")

✓ VectorStore defined


## 8. RAG Chatbot

(from the original `chatbot.py`, **fixed**)

Changes from the original:
- Removed `from langchain.chains import RetrievalQA` — this is what caused
  `ModuleNotFoundError: No module named 'langchain.chains'`. `RetrievalQA` is
  also deprecated upstream in favor of LCEL chains.
- The QA chain is now built with LCEL (`retriever | format_docs`, prompt,
  llm, output parser) using only `langchain-core`, so it no longer depends
  on the full `langchain` umbrella package being correctly resolved.
- `ask()` now retrieves source documents and the answer separately, since
  there's no `RetrievalQA` wrapper bundling them together anymore.


In [10]:
class RAGChatbot:
    """RAG-based chatbot using Jina Embeddings and Groq LLM"""

    PROMPT_TEMPLATE = """You are a helpful assistant answering questions based on provided documents.

Context information from documents:
{context}

Question: {question}

Provide a detailed and accurate answer based on the context. If the information is not in the context, say so clearly.

Answer:"""

    def __init__(self, vector_db_path: str = "./vector_store"):
        self.vector_store_manager = VectorStore(vector_db_path)
        self.llm = self._initialize_llm()
        self.qa_chain = None
        self.retriever = None
        self.prompt = PromptTemplate(
            template=self.PROMPT_TEMPLATE,
            input_variables=["context", "question"],
        )

    def _initialize_llm(self):
        """Initialize Groq LLM"""
        groq_api_key = os.getenv("GROQ_API_KEY")
        if not groq_api_key:
            raise ValueError("GROQ_API_KEY not found in environment variables")

        llm = ChatGroq(
            groq_api_key=groq_api_key,
            model_name=GROQ_MODEL,
            temperature=LLM_CONFIG["temperature"],
            max_tokens=LLM_CONFIG["max_tokens"],
        )
        print("✓ Groq LLM initialized")
        return llm

    def setup_from_pdfs(self, pdf_folder_path: str = "./pdfs"):
        """Setup RAG system from PDF files"""
        print("=" * 60)
        print("RAG CHATBOT - INITIALIZATION")
        print("=" * 60)

        pdf_loader = PDFLoader(pdf_folder_path)
        documents = pdf_loader.process_pdfs()

        if not documents:
            raise ValueError("No documents loaded from PDFs")

        self.vector_store_manager.create_vector_store(documents)
        self.vector_store_manager.save_vector_store()

        self._setup_qa_chain()

        print("\n" + "=" * 60)
        print("✓ RAG system ready!")
        print("=" * 60)

    def load_existing_vectorstore(self):
        """Load existing vector store"""
        print("Loading existing vector store...")
        self.vector_store_manager.load_vector_store()
        self._setup_qa_chain()
        print("✓ Vector store loaded and RAG system ready!")

    @staticmethod
    def _format_docs(docs: List[Document]) -> str:
        return "\n\n".join(doc.page_content for doc in docs)

    def _setup_qa_chain(self):
        """Setup the retrieval chain using LCEL (no langchain.chains needed)"""
        vector_store = self.vector_store_manager.vector_store
        self.retriever = vector_store.as_retriever(search_kwargs=RETRIEVER_CONFIG["search_kwargs"])

        self.qa_chain = (
            {
                "context": self.retriever | self._format_docs,
                "question": RunnablePassthrough(),
            }
            | self.prompt
            | self.llm
            | StrOutputParser()
        )

    def ask(self, query: str) -> dict:
        """Ask a question to the chatbot"""
        if self.qa_chain is None:
            raise ValueError("RAG system not initialized. Call setup_from_pdfs() first")

        print(f"\n📝 Query: {query}")
        print("-" * 60)

        source_documents = self.retriever.invoke(query)
        answer = self.qa_chain.invoke(query)

        result = {"result": answer, "source_documents": source_documents}

        print(f"\n✓ Answer:\n{result['result']}")

        print("\n📚 Source Documents:")
        for i, doc in enumerate(result["source_documents"], 1):
            print(f"\n[Source {i}]")
            print(f"Page: {doc.metadata.get('page', 'Unknown')}")
            print(f"Content: {doc.page_content[:200]}...")

        return result

    def interactive_chat(self):
        """Start interactive chat session"""
        if self.qa_chain is None:
            raise ValueError("RAG system not initialized. Call setup_from_pdfs() first")

        print("\n" + "=" * 60)
        print("INTERACTIVE CHAT MODE")
        print("Type 'exit' to quit")
        print("=" * 60)

        while True:
            try:
                user_input = input("\n❓ Your question: ").strip()

                if user_input.lower() == "exit":
                    print("\n👋 Goodbye!")
                    break

                if not user_input:
                    print("⚠️  Please enter a question")
                    continue

                self.ask(user_input)

            except KeyboardInterrupt:
                print("\n👋 Chat interrupted. Goodbye!")
                break
            except Exception as e:
                print(f"❌ Error: {e}")

print("✓ RAGChatbot defined")

✓ RAGChatbot defined


## 9. Initialize the chatbot

This replaces `main.py`. It checks for an existing vector store and either
loads it or builds a new one from the PDFs in `PDF_FOLDER_PATH`.

In [11]:
chatbot = RAGChatbot(vector_db_path=VECTOR_DB_PATH)

vector_store_exists = os.path.exists(VECTOR_DB_PATH) and any(Path(VECTOR_DB_PATH).iterdir())

if vector_store_exists:
    print("✓ Vector store found, loading it...")
    chatbot.load_existing_vectorstore()
else:
    print(f"No existing vector store found. Building one from PDFs in: {PDF_FOLDER_PATH}")
    chatbot.setup_from_pdfs(PDF_FOLDER_PATH)

✓ Groq LLM initialized
✓ Vector store found, loading it...
Loading existing vector store...
Loading vector store from ./vector_store...
✓ Vector store loaded
✓ Vector store loaded and RAG system ready!


## 10. Ask a single question

This replaces `query.py`'s `run_query()`. Edit the question below and re-run.

In [12]:
# result = chatbot.ask("What is the main topic of the document?")

## 11. Batch questions (optional)

This replaces `query.py`'s `batch_queries()`.

In [ ]:
# example_queries = [
#     "What is the main topic of the document?",
#     "Summarize the key points",
#     "What are the important concepts mentioned?",
# ]

# batch_results = []
# for i, q in enumerate(example_queries, 1):
#     print(f"\n[Query {i}/{len(example_queries)}]")
#     batch_results.append(chatbot.ask(q))

## 12. Interactive chat (optional)

Run this cell to chat with the bot directly in the notebook. Type `exit` to
stop. (Jupyter's `input()` works fine here — a prompt box will appear below
the cell.)

In [ ]:
chatbot.interactive_chat()


INTERACTIVE CHAT MODE
Type 'exit' to quit

📝 Query: what is the codeprolk
------------------------------------------------------------

✓ Answer:
CodePRO LK is a dynamic educational platform that offers a diverse range of technology-related courses in Sinhala, aimed at empowering Sri Lankans with valuable skills in programming, data science, and machine learning. It was founded by Dinesh Piyasamara during the COVID-19 pandemic to provide Sri Lankan students with the tools and knowledge to thrive in a digitally-driven world, all through their native language.

The platform provides a holistic learning experience, comprising video lectures, interactive quizzes, and hands-on assignments. It also has a vibrant community where learners can interact, share insights, and support each other, as well as consultation services for personalized learning support.

CodePRO LK's vision is to assist talented Sri Lankans in reaching the international market by providing a solid foundation in technolog